<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Video to Gaussian Splat with Diffusers, VGGT, and gsplat

This notebook is the cookbook form of the standalone `demos/cosmos3_gaussian_splat` project. It generates a camera-conditioned Cosmos3 video from one image, estimates the geometry actually present in the generated frames with VGGT, and optimizes a Gaussian splat.

## 1. Prerequisites

- Linux and an NVIDIA GPU; one A100 80 GB is the validated target.
- `uv` and Python 3.11 or 3.12.
- Access to `nvidia/Cosmos3-Nano` and `nvidia/Cosmos-1.0-Guardrail`.
- `HF_TOKEN` supplied through environment or secret management—never pasted into this notebook.

The CPU-safe cells below inspect configuration and camera actions. GPU stages are explicitly gated by `RUN_GPU_STAGES`.

In [ ]:
from pathlib import Path
import os
import sys


def find_cosmos_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "cookbooks").is_dir() and (path / "demos/cosmos3_gaussian_splat").is_dir():
            return path
    raise FileNotFoundError("Run this notebook inside a Cosmos checkout")


COSMOS_ROOT = find_cosmos_root(Path.cwd().resolve())
STANDALONE_ROOT = COSMOS_ROOT / "demos/cosmos3_gaussian_splat"
OUTPUT_DIR = Path(os.environ.get("COSMOS3_GSPLAT_OUTPUT", STANDALONE_ROOT / "outputs/notebook-test")).resolve()
REFERENCE_IMAGE_VALUE = os.environ.get("COSMOS3_GSPLAT_IMAGE")
MASK_VALUE = os.environ.get("COSMOS3_GSPLAT_MASK")
REFERENCE_IMAGE = Path(REFERENCE_IMAGE_VALUE).expanduser().resolve() if REFERENCE_IMAGE_VALUE else None
OBJECT_MASK = Path(MASK_VALUE).expanduser().resolve() if MASK_VALUE else None
PROMPT = os.environ.get(
    "COSMOS3_GSPLAT_PROMPT",
    "A stationary chair while the camera moves smoothly around it and the lighting remains fixed.",
)
RUN_GPU_STAGES = os.environ.get("COSMOS3_GSPLAT_RUN_GPU", "false").lower() in {"1", "true", "yes"}

if str(STANDALONE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT / "src"))

print("Cosmos root:", COSMOS_ROOT)
print("Standalone project:", STANDALONE_ROOT)
print("Output:", OUTPUT_DIR)
print("Reference:", REFERENCE_IMAGE or "<set COSMOS3_GSPLAT_IMAGE>")
print("GPU stages:", RUN_GPU_STAGES)

## 2. Install the standalone environment

Run this once in a terminal on the GPU host:

```bash
cd demos/cosmos3_gaussian_splat
uv sync --frozen --extra gpu
```

Then select that environment as the notebook kernel, or start Jupyter through `uv run`. The notebook does not mutate environments automatically.

In [ ]:
import numpy as np

from cosmos3_gsplat import Cosmos3GaussianSplatPipeline, PipelineConfig
from cosmos3_gsplat.trajectory import make_closed_helical_trajectory

config = PipelineConfig(profile="test", prompt=PROMPT)
trajectory = make_closed_helical_trajectory(config.trajectory)

print("poses:", trajectory.poses_c2w.shape)
print("actions:", trajectory.raw_actions.shape)
print("azimuth span:", float(trajectory.azimuth_deg[-1] - trajectory.azimuth_deg[0]), "degrees")
print("elevation range:", float(trajectory.elevation_deg.min()), "to", float(trajectory.elevation_deg.max()))
print("loop closure max error:", float(np.abs(trajectory.poses_c2w[0] - trajectory.poses_c2w[-1]).max()))

In [ ]:
try:
    import matplotlib.pyplot as plt

    centers = trajectory.poses_c2w[:, :3, 3]
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(centers[:, 0], centers[:, 1], marker=".")
    axes[0].set_title("Commanded path: top view")
    axes[0].set_xlabel("X")
    axes[0].set_ylabel("Y")
    axes[0].axis("equal")
    axes[1].plot(trajectory.azimuth_deg, trajectory.elevation_deg)
    axes[1].set_title("Elevation over azimuth")
    axes[1].set_xlabel("Azimuth (degrees)")
    axes[1].set_ylabel("Elevation (degrees)")
    plt.show()
except ImportError:
    print("Install the GPU extra to display the matplotlib trajectory plot.")

## 3. Generate the Cosmos video

This stage sends the reference image, prompt, and 60 camera-pose actions to `Cosmos3OmniPipeline`. Cosmos returns 61 RGB frames. The commanded transforms are saved separately because Cosmos does not return calibrated cameras and may only approximate the requested motion.

Inspect `generated/cosmos_orbit.mp4` before proceeding. A successful reconstruction needs real viewpoint change—not texture breathing or a near-static wobble.

In [ ]:
pipeline = Cosmos3GaussianSplatPipeline.from_pretrained(
    config.generation.model_id,
    geometry_model=config.geometry.model_id,
    config=config,
)

if RUN_GPU_STAGES:
    if REFERENCE_IMAGE is None or not REFERENCE_IMAGE.is_file():
        raise FileNotFoundError("Set COSMOS3_GSPLAT_IMAGE to a reference image")
    generation_result = pipeline(
        prompt=PROMPT,
        image=REFERENCE_IMAGE,
        object_mask=OBJECT_MASK,
        trajectory=trajectory,
        output_dir=OUTPUT_DIR,
        stages=("generate",),
    )
    print("Generated video:", generation_result.generated_video)
else:
    print("Skipped. Set COSMOS3_GSPLAT_RUN_GPU=true on a CUDA host to run generation.")

## 4. Recover apparent geometry with VGGT

VGGT jointly estimates camera extrinsics, shared intrinsics, dense depth, and confidence from selected generated frames. Its world-to-camera estimates are inverted and robustly Sim(3)-aligned with the commanded camera-to-world path.

Command residuals are a quality metric and soft prior. The generated pixels—not the requested path—determine the geometry that can actually be reconstructed.

In [ ]:
if RUN_GPU_STAGES:
    geometry_result = pipeline(
        prompt=PROMPT,
        image=REFERENCE_IMAGE,
        object_mask=OBJECT_MASK,
        trajectory=trajectory,
        output_dir=OUTPUT_DIR,
        stages=("geometry",),
    )
    geometry_metrics = __import__("json").loads((OUTPUT_DIR / "geometry/vggt/geometry.json").read_text())["metrics"]
    for key in [
        "accepted_views",
        "views_within_command_bounds",
        "pose_prior_reliable",
        "rotation_residual_deg_median",
        "predicted_loop_closure_translation",
    ]:
        print(f"{key}: {geometry_metrics.get(key)}")
else:
    print("Skipped VGGT geometry stage.")

## 5. Optimize and export the Gaussian splat

Proceed only when the generated video visibly contains wide-baseline parallax and VGGT does not report catastrophic command disagreement. The test profile runs 100 optimization steps; switch to `PipelineConfig(profile="full")` only after the test result passes that gate.

The final stage exports:

- `splat/gaussian_splat.ply` — interoperable Gaussian-splat PLY;
- `splat/gaussian_splat.splat` — compact viewer format;
- `splat/splat.ply` — backward-compatible alias;
- comparison/render videos and an HTML report.

In [ ]:
if RUN_GPU_STAGES:
    force_splat = os.environ.get("COSMOS3_GSPLAT_FORCE_SPLAT", "false").lower() in {"1", "true", "yes"}
    quality_gate_passed = bool(geometry_metrics.get("pose_prior_reliable"))
    if quality_gate_passed or force_splat:
        splat_result = pipeline(
            prompt=PROMPT,
            image=REFERENCE_IMAGE,
            object_mask=OBJECT_MASK,
            trajectory=trajectory,
            output_dir=OUTPUT_DIR,
            stages=("splat", "report"),
        )
        print("Gaussian PLY:", splat_result.splat_path)
        print("Report:", splat_result.report_path)
    else:
        print("Stopped before splat training: generated cameras did not agree with the commanded path.")
        print("Inspect the video and set COSMOS3_GSPLAT_FORCE_SPLAT=true only for diagnostic output.")
else:
    print("Skipped splat and report stages.")

## 6. Inspect outputs

Open `report/index.html` for trajectory plots, metrics, generated video, refined render, and generated-versus-splat comparison. When the pipeline runs through Hugging Face Jobs, the same directory tree is persisted under `hf://buckets/<namespace>/<bucket>/runs/<run-id>/`.

In [ ]:
if OUTPUT_DIR.exists():
    from IPython.display import Video, display

    for label, path in [
        ("Cosmos generation", OUTPUT_DIR / "generated/cosmos_orbit.mp4"),
        ("Gaussian render", OUTPUT_DIR / "splat/render_orbit.mp4"),
        ("Generated vs splat", OUTPUT_DIR / "splat/generated_vs_splat.mp4"),
    ]:
        if path.is_file():
            print(label, path)
            display(Video(str(path), embed=True))
else:
    print("No output directory yet.")

## 7. A100 alternative with persistent HF Bucket artifacts

From `demos/cosmos3_gaussian_splat`:

```bash
hf auth login
uv run cosmos3-gsplat-submit \
  --reference-image /path/to/chair.png \
  --mask /path/to/chair_mask.png \
  --prompt "A stationary chair while the camera moves around it." \
  --bucket <namespace>/cosmos3-gsplat-artifacts \
  --profile full \
  --wait \
  --download-dir ./outputs/chair-full
```

Use a fine-grained token through environment/secret management. The Job creates an A100 run, stores all intermediates and final splats in the bucket, and writes `complete.json` only after every stage succeeds.